# E2 — validation against measured human abundance

The frozen folding head was trained on MegaScale proteolysis with the **entire SH3 family held
out**. Human Domainome 1.0 measured the cellular abundance of variants in human SH3 domains by
aPCA — a different assay, a different laboratory, a different observable.

Where the two overlap we get a genuine external test, on **real COSMIC cancer variants**, with no
wet lab of our own.

Two questions:

1. **Does the head predict measured abundance?** Per domain and pooled.
2. **How well *could* anything do?** Seven SH3 domains were measured by *both* MegaScale and
   Domainome. Their agreement is the ceiling: no predictor can beat two careful experiments
   agreeing with each other. A correlation is meaningless without it.

### Signs — the trap in this program

| Quantity | Convention |
|---|---|
| `pred_ddG_fold` (ours) | **positive** = destabilising |
| `normalized_fitness` (aPCA) | **negative** = destabilising |
| Rocklin ΔΔG, column V35 | **negative** = destabilising |

So ours correlates **negatively** with fitness, and the two assays correlate **positively** with
each other. Everything is reported as |ρ|. Getting this backwards silently inverts the result —
see `reference/domainome_column_map.md`.

Runtime: CPU. About 3 minutes. Reads the frozen run; never writes to it.

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, hashlib
from pathlib import Path
import pandas as pd, numpy as np
from scipy.stats import spearmanr, mannwhitneyu

DRIVE = Path('/content/drive/MyDrive')

def pick(*names, what=''):
    for n in names:
        if (DRIVE / n).exists(): return DRIVE / n
    raise FileNotFoundError(f'No {what} folder in MyDrive. Tried: ' + ', '.join(names))

PROJECT = pick('sh3_repro/extended', 'SH3_extended', 'SH3 extended', what='project')
FROZEN  = pick('sh3_repro', what='frozen-run')          # READ ONLY
IN      = next((PROJECT / n for n in ('input/external', 'input_new')
                if (PROJECT / n).exists()), PROJECT / 'input/external')
FROZEN_IN = PROJECT / 'input' / 'frozen'
DOM     = IN / 'domainome'
OUT     = PROJECT / 'results' / 'E2_domainome_external'
OUT.mkdir(parents=True, exist_ok=True)

def frozen(rel, bundled):
    """Read from the frozen run, or from the bundled copy in input/frozen/."""
    p = FROZEN / 'results' / rel
    return p if p.exists() else FROZEN_IN / 'results' / bundled

print('project:', PROJECT); print('frozen :', FROZEN)

## 1. Load and deduplicate

Two defects in the raw Domainome table, both found in X0 and repeated here so this notebook
stands alone:

- 45 rows carry a fitness value but a **null position**, so they cannot join.
- One physical domain can hold **two Pfam annotations at the same start** — ITSN1 has both
  `PF00018_1003` and `PF07653_1003` — so every variant in it appears twice. Left uncollapsed it
  duplicates atlas rows in the merge.

In [ ]:
SH3_PFAM = ('PF00018', 'PF07653', 'PF14604')      # SH3_1, SH3_2, SH3_9
KEY = ['uniprot_ID', 'position', 'wt_aa', 'mut_aa']

dom = pd.read_csv(DOM / 'Extended_data_Table_2_fitness_scores_normalized_domainranks.txt', sep='\t')
dom['STOP'] = dom.STOP.astype('string').str.upper()   # TRUE/FALSE may parse as bool
sh3 = dom[dom.domain_ID.str.contains('|'.join(SH3_PFAM))]
sh3 = sh3[(sh3.STOP == 'FALSE') & sh3.normalized_fitness.notna()].copy()
n0 = len(sh3)

sh3 = sh3.dropna(subset=['position', 'wt_aa', 'mut_aa'])
sh3['position'] = sh3.position.astype(int)
n_null = n0 - len(sh3)
n_dup = int(sh3.duplicated(KEY, keep=False).sum())
sh3 = sh3.sort_values('quality_rank').drop_duplicates(KEY, keep='first')
assert not sh3.duplicated(KEY).any()
del dom

print(f'{n0:,} SH3 rows -> dropped {n_null} with a null key, '
      f'collapsed {n_dup:,} duplicated by dual Pfam annotation')
print(f'{len(sh3):,} unique variants in {sh3.domain_ID.nunique()} domains')

In [ ]:
# the frozen predictions, and the atlas for the protein position
pred  = pd.read_parquet(frozen('B3_uncertainty_and_calls/atlas_predictions.parquet',
                               'atlas_predictions.parquet'))
atlas = pd.read_csv(frozen('A0_cancer_atlas/SH3_cancer_master.tsv', 'SH3_cancer_master.tsv'),
                    sep='\t', usecols=['variant_id', 'pos_protein'])
pred = pred.merge(atlas, on='variant_id', how='left')
N_ATLAS = len(pred)
print(f'{N_ATLAS:,} atlas variants with frozen predictions')

## 2. How many cancer variants were actually measured?

In [ ]:
DCOLS = KEY + ['domain_ID', 'normalized_fitness', 'normalized_fitness_sigma', 'quality_rank']
L, R = ['uniprot','pos_protein','wt_aa','mut_aa'], ['uniprot_ID','position','wt_aa','mut_aa']

full = pred.merge(sh3[DCOLS], left_on=L, right_on=R, how='left')
assert len(full) == N_ATLAS, f'merge expanded {N_ATLAS} -> {len(full)}'   # must never grow
m = pred.merge(sh3[DCOLS], left_on=L, right_on=R, how='inner')

print(f'{len(m)} of {N_ATLAS:,} atlas cancer variants have a direct Domainome measurement')
print(f'  across {m.domain_ID.nunique()} domains and {m.gene.nunique()} genes')

## 3. Does the head predict measured abundance?

Negative ρ is the correct direction: high predicted ΔΔG (destabilising) → low abundance.

In [ ]:
rho, p = spearmanr(m.pred_ddG_fold, m.normalized_fitness)
print(f'POOLED  rho = {rho:+.3f}   (p = {p:.2e}, n = {len(m)})')

rows = []
for d, g in m.groupby('domain_ID'):
    if len(g) < 20: continue
    r, pp = spearmanr(g.pred_ddG_fold, g.normalized_fitness)
    rows.append((d, g.gene.iloc[0], len(g), r, pp))
per_dom = pd.DataFrame(rows, columns=['domain','gene','n','rho','p']).sort_values('rho')
per_dom.to_csv(OUT / 'per_domain_E2.csv', index=False)
m.to_csv(OUT / 'matched_variants_E2.csv', index=False)

print(f'\nPER DOMAIN (n>=20): {len(per_dom)} domains, median rho {per_dom.rho.median():+.3f}, '
      f'correct direction in {(per_dom.rho < 0).sum()}/{len(per_dom)}')
print(per_dom[['gene','n','rho']].to_string(index=False))

## 4. Do the confidently destabilising calls separate?

In [ ]:
m['destab'] = m.fold_lo >= 1
a, b = m.loc[m.destab, 'normalized_fitness'], m.loc[~m.destab, 'normalized_fitness']
u, p_mw = mannwhitneyu(a, b, alternative='less')
print(f'destabilising  n={len(a):4}  median fitness {a.median():+.3f}')
print(f'tolerated      n={len(b):4}  median fitness {b.median():+.3f}')
print(f'Mann-Whitney (destab < tolerated): p = {p_mw:.3e}')

## 5. The noise ceiling

Seven SH3 domains were measured by **both** studies. Their agreement bounds what any model can
reach against Domainome fitness.

**Position mapping.** Rocklin positions are domain-local (1–70); Domainome's are absolute protein
positions. The conversion is

```
rocklin_pos = domainome_position − domain_start + 1 + offset
```

with `domain_start` the suffix of `domain_ID` and `offset` from the overlaps file. Verified
empirically: for BTK this gives 1,102 matches at the recorded offset against ≤114 at any other
shift. Joining on absolute positions gives **zero** matches — an easy and silent mistake.

In [ ]:
OV = ['pdb_id','uniprot_ID','dom_ID','aa_seq_aPCA','aa_seq_Rocklin',
      'overlap','offset','percent_overlap']        # the file has no header
GENE = {'Q06187':'BTK','P08631':'HCK','P06239':'LCK','O15259':'NPHP1',
        'P20936':'RASA1','P07751':'SPTB1'}

ov = pd.read_csv(DOM / 'analysis_files/domains_from_tsuboyama_rocklin_overlaps.txt',
                 sep='\t', header=None, names=OV)
ov = ov[ov.dom_ID.str.contains('|'.join(SH3_PFAM))].copy()
ov['start'] = ov.dom_ID.str.rsplit('_', n=1).str[1].astype(int)
print(f'{len(ov)} SH3 domain pairs measured in both studies')

# headerless: V29 mutation code, V30 pdb name, V35 ddG  (R 1-indexed -> 28, 29, 34 here)
roc = pd.read_csv(DOM / 'analysis_files/K50_dG_Dataset1_Dataset2_in_aPCA.csv',
                  header=None, low_memory=False)
roc['pdb_id'] = roc[29].astype(str).str.split('.pdb').str[0]
mm = roc[28].astype(str).str.extract(r'^([A-Z])(\d+)([A-Z])$')
roc = roc.assign(wt_aa=mm[0], rpos=pd.to_numeric(mm[1], errors='coerce'), mut_aa=mm[2],
                 rocklin_ddG=pd.to_numeric(roc[34], errors='coerce'))
roc = roc.dropna(subset=['rpos','rocklin_ddG']).merge(ov, on='pdb_id')

d = sh3.rename(columns={'domain_ID':'dom_ID'}).merge(
        ov[['dom_ID','offset','percent_overlap']], on='dom_ID')
d['rpos'] = (d.position - d.dom_ID.str.rsplit('_', n=1).str[1].astype(int) + 1 + d.offset).astype(int)

j = roc.merge(d[['dom_ID','rpos','wt_aa','mut_aa','normalized_fitness',
                 'mean_input_count','uniprot_ID','percent_overlap']],
              on=['dom_ID','rpos','wt_aa','mut_aa'], how='inner', suffixes=('','_d'))
j = j[(j.percent_overlap > 80) & (j.mean_input_count > 5)]      # the authors' own QC
print(f'{len(j):,} variants measured by both assays after QC')

In [ ]:
ceil = []
for dm, g in j.groupby('dom_ID'):
    if len(g) < 20: continue
    r, pp = spearmanr(g.rocklin_ddG, g.normalized_fitness)
    o = per_dom.set_index('domain').rho.get(dm, np.nan)
    ceil.append((GENE.get(g.uniprot_ID.iloc[0], g.uniprot_ID.iloc[0]), dm, len(g),
                 abs(r), pp, abs(o) if pd.notna(o) else np.nan))
t = pd.DataFrame(ceil, columns=['gene','domain','n','ceiling','p','ours']).sort_values('ceiling', ascending=False)
t['pct_of_ceiling'] = 100 * t.ours / t.ceiling
t.to_csv(OUT / 'noise_ceiling_E2.csv', index=False)
print('|rho| throughout. ceiling = MegaScale vs Domainome on the same variants.')
print(t[['gene','n','ceiling','ours','pct_of_ceiling']].to_string(index=False))
print(f'\nmedian ceiling {t.ceiling.median():.3f}')

### Matched subset

The table above compares unequal samples: the ceiling uses ~1,000 variants per domain, our figure
the cancer-variant subset. Restricting both sides to **exactly the same variants** makes them
directly comparable — at the cost of small n.

In [ ]:
j2 = j.merge(m[['domain_ID','pos_protein','wt_aa','mut_aa','pred_ddG_fold']],
             left_on=['dom_ID','wt_aa','mut_aa'],
             right_on=['domain_ID','wt_aa','mut_aa'], how='inner')
j2 = j2[j2.rpos == (j2.pos_protein - j2.dom_ID.str.rsplit('_', n=1).str[1].astype(int) + 1 + j2.offset)]

rows = []
for dm, g in j2.groupby('dom_ID'):
    if len(g) < 15: continue
    c = abs(spearmanr(g.rocklin_ddG, g.normalized_fitness)[0])
    o = abs(spearmanr(g.pred_ddG_fold, g.normalized_fitness)[0])
    rows.append((GENE.get(g.uniprot_ID.iloc[0], '?'), dm, len(g), c, o, 100*o/c))
t2 = pd.DataFrame(rows, columns=['gene','domain','n','ceiling','ours','pct_of_ceiling'])
t2.to_csv(OUT / 'noise_ceiling_matched_E2.csv', index=False)
print('identical variants both sides:')
print(t2[['gene','n','ceiling','ours','pct_of_ceiling']].to_string(index=False))

## 6. RESULTS_E2.md and metrics_E2.json

Machine-written. Nothing below is typed by hand; the interpretation lives in `ANALYSIS_E2.md`.

In [ ]:
metrics = {
 'program': 'E2', 'date': time.strftime('%Y-%m-%d %H:%M'),
 'atlas_variants': int(N_ATLAS),
 'measured_in_domainome': int(len(m)),
 'domains': int(m.domain_ID.nunique()), 'genes': int(m.gene.nunique()),
 'pooled_rho': float(rho), 'pooled_p': float(p),
 'per_domain_median_rho': float(per_dom.rho.median()),
 'per_domain_correct_direction': f'{int((per_dom.rho<0).sum())}/{len(per_dom)}',
 'destabilising_n': int(len(a)), 'destabilising_median_fitness': float(a.median()),
 'tolerated_n': int(len(b)), 'tolerated_median_fitness': float(b.median()),
 'mannwhitney_p': float(p_mw),
 'ceiling_median': float(t.ceiling.median()),
 'ceiling_domains': int(len(t)),
 'domainome_rows_dropped_null_key': int(n_null),
 'domainome_rows_collapsed_dual_pfam': int(n_dup),
}
(OUT / 'metrics_E2.json').write_text(json.dumps(metrics, indent=1))

L = ['# E2 - validation against measured human abundance', '',
     f"Run {metrics['date']}. Machine-written; see ANALYSIS_E2.md for interpretation.", '',
     '## Coverage', '',
     f"| atlas variants | {N_ATLAS:,} |", '|---|---|',
     f"| measured in Domainome | **{len(m)}** |",
     f"| domains / genes | {m.domain_ID.nunique()} / {m.gene.nunique()} |", '',
     '## Agreement with measured abundance', '',
     f"Pooled Spearman rho = **{rho:+.3f}** (p = {p:.2e}, n = {len(m)}).",
     f"Per-domain median {per_dom.rho.median():+.3f}; correct direction in "
     f"{int((per_dom.rho<0).sum())}/{len(per_dom)} domains with n >= 20.", '',
     '| gene | n | rho |', '|---|---|---|']
L += [f'| {r.gene} | {r.n} | {r.rho:+.3f} |' for r in per_dom.itertuples()]
L += ['', '## Confidently destabilising calls (fold_lo >= 1)', '',
      f"median fitness {a.median():+.3f} (n={len(a)}) against {b.median():+.3f} (n={len(b)}); "
      f"Mann-Whitney p = {p_mw:.3e}.", '',
      '## Noise ceiling', '',
      'MegaScale vs Domainome on the same variants. |rho| throughout.', '',
      '| gene | n | ceiling | ours | share |', '|---|---|---|---|---|']
L += [f'| {r.gene} | {r.n} | {r.ceiling:.3f} | ' +
      (f'{r.ours:.3f} | {r.pct_of_ceiling:.0f}% |' if pd.notna(r.ours) else '- | - |')
      for r in t.itertuples()]
L += ['', f'Median ceiling **{t.ceiling.median():.3f}**.', '',
      '### Matched subset (identical variants both sides)', '',
      '| gene | n | ceiling | ours | share |', '|---|---|---|---|---|']
L += [f'| {r.gene} | {r.n} | {r.ceiling:.3f} | {r.ours:.3f} | {r.pct_of_ceiling:.0f}% |'
      for r in t2.itertuples()]
L += ['', '## Data handling', '',
      f'- {n_null} Domainome rows dropped: fitness present but a null position.',
      f'- {n_dup:,} rows collapsed: one domain can carry two Pfam annotations at the same start.',
      '- The left join is asserted not to change the atlas row count.']
(OUT / 'RESULTS_E2.md').write_text('\n'.join(L))
print('\n'.join(L))

In [ ]:
print('written to', OUT)
for f in sorted(OUT.iterdir()): print(f'  {f.stat().st_size:>9,} B  {f.name}')